<a href="https://colab.research.google.com/github/Ad450/llm-interpretability/blob/main/resnet_attack.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import json, os, random, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, models, transforms
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
LAYERS = ["stem", "layer1", "layer2", "layer3", "layer4"]
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", dev)

device: cpu


In [3]:
SYNTHETIC = True          # True = smoke test with fake data; False = real GTSRB run
OUT = "smoke" if SYNTHETIC else "results"
DATA_ROOT = "data"
IMG = 64
EPOCHS = 1 if SYNTHETIC else 3
BS = 128
LR = 3e-4
N_EVAL = 200 if SYNTHETIC else 2000          # test images to attack
N_ANALYSIS = 100 if SYNTHETIC else 1000      # clean-correct images for layerwise analysis
EPS_255 = [1, 2, 4, 8]                       # attack sizes, in units of 1/255
PGD_STEPS = 10
SEED = 0

os.makedirs(OUT, exist_ok=True)
EPS_LIST = [e / 255.0 for e in EPS_255]

random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

In [4]:
class Net(nn.Module):
    """ResNet-18 that takes [0,1] images and normalises internally."""
    def __init__(self, num_classes=43, pretrained=True):
        super().__init__()
        weights = models.ResNet18_Weights.DEFAULT if pretrained else None
        r = models.resnet18(weights=weights)
        r.fc = nn.Linear(r.fc.in_features, num_classes)
        self.r = r
        self.register_buffer("mean", torch.tensor(IMAGENET_MEAN).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor(IMAGENET_STD).view(1, 3, 1, 1))

    def forward(self, x):
        return self.r((x - self.mean) / self.std)

    def hook_points(self):
        r = self.r
        return {"stem": r.maxpool, "layer1": r.layer1, "layer2": r.layer2,
                "layer3": r.layer3, "layer4": r.layer4}

In [5]:
def synthetic_data(n_train, n_test, img, num_classes=43, seed=0):
    g = torch.Generator().manual_seed(seed)
    def make(n):
        y = torch.randint(0, num_classes, (n,), generator=g)
        x = torch.rand(n, 3, img, img, generator=g) * 0.2
        for i in range(n):
            c = int(y[i])
            x[i, c % 3] += 0.5
            x[i, :, (c * 3) % img:(c * 3) % img + 4, :] += 0.3
        return x.clamp(0, 1), y
    return make(n_train), make(n_test)


def load_gtsrb(root, img, n_eval, seed):
    tf = transforms.Compose([transforms.Resize((img, img)), transforms.ToTensor()])
    train = datasets.GTSRB(root=root, split="train", download=True, transform=tf)
    test = datasets.GTSRB(root=root, split="test", download=True, transform=tf)

    def to_tensors(ds, idx):
        xs, ys = [], []
        for i in idx:
            x, y = ds[int(i)]
            xs.append(x); ys.append(y)
        return torch.stack(xs), torch.tensor(ys)

    rng = np.random.RandomState(seed)
    train_xy = to_tensors(train, range(len(train)))
    test_idx = rng.permutation(len(test))[:n_eval]
    test_xy = to_tensors(test, test_idx)
    return train_xy, test_xy

In [6]:
if SYNTHETIC:
    train_xy, test_xy = synthetic_data(1000, N_EVAL, IMG, seed=SEED)
else:
    train_xy, test_xy = load_gtsrb(DATA_ROOT, IMG, N_EVAL, SEED)
x_test, y_test = test_xy
print("train:", len(train_xy[0]), " eval subset:", len(x_test))

train: 1000  eval subset: 200


In [7]:
def train(model, train_xy, epochs, bs, lr):
    x, y = train_xy
    loader = DataLoader(TensorDataset(x, y), batch_size=bs, shuffle=True)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    for ep in range(epochs):
        model.train()
        tot, correct, loss_sum = 0, 0, 0.0
        for xb, yb in loader:
            xb, yb = xb.to(dev), yb.to(dev)
            opt.zero_grad()
            out = model(xb)
            loss = F.cross_entropy(out, yb)
            loss.backward()
            opt.step()
            tot += len(yb)
            correct += (out.argmax(1) == yb).sum().item()
            loss_sum += loss.item() * len(yb)
        print(f"epoch {ep + 1}/{epochs}  loss {loss_sum / tot:.4f}  train acc {correct / tot:.4f}")
    model.eval()


@torch.no_grad()
def predict(model, x, bs=BS):
    model.eval()
    return torch.cat([model(x[i:i + bs].to(dev)).argmax(1).cpu() for i in range(0, len(x), bs)])

In [8]:
model = Net(pretrained=not SYNTHETIC).to(dev)
ckpt = os.path.join(OUT, "resnet18_gtsrb.pt")

if os.path.exists(ckpt):
    model.load_state_dict(torch.load(ckpt, map_location=dev))
    model.eval()
    print("loaded checkpoint")
else:
    t0 = time.time()
    train(model, train_xy, EPOCHS, BS, LR)
    torch.save(model.state_dict(), ckpt)
    print(f"trained in {time.time() - t0:.0f}s")

for p in model.parameters():
    p.requires_grad_(False)

epoch 1/1  loss 1.7375  train acc 0.6640
trained in 27s


In [9]:
def fgsm(model, x, y, eps):
    x = x.clone().requires_grad_(True)
    loss = F.cross_entropy(model(x), y)
    (g,) = torch.autograd.grad(loss, x)
    return (x + eps * g.sign()).clamp(0, 1).detach()


def pgd(model, x, y, eps, steps, alpha):
    xa = (x + torch.empty_like(x).uniform_(-eps, eps)).clamp(0, 1)
    for _ in range(steps):
        xa.requires_grad_(True)
        loss = F.cross_entropy(model(xa), y)
        (g,) = torch.autograd.grad(loss, xa)
        xa = xa.detach() + alpha * g.sign()
        xa = torch.min(torch.max(xa, x - eps), x + eps).clamp(0, 1)
    return xa.detach()


def run_attack(model, x, y, name, eps, steps=PGD_STEPS, bs=BS):
    outs = []
    for i in range(0, len(x), bs):
        xb, yb = x[i:i + bs].to(dev), y[i:i + bs].to(dev)
        if name == "fgsm":
            xa = fgsm(model, xb, yb, eps)
        else:
            xa = pgd(model, xb, yb, eps, steps, alpha=2.5 * eps / steps)
        outs.append(xa.cpu())
    return torch.cat(outs)

In [10]:
@torch.no_grad()
def get_acts(model, x, bs=BS):
    """Returns flat activations per layer, pooled activations per layer, and predictions."""
    flat = {k: [] for k in LAYERS}
    pooled = {k: [] for k in LAYERS}
    handles = []

    def make_hook(name):
        def hook(_m, _i, o):
            pooled[name].append(o.mean(dim=(2, 3)).cpu())
            flat[name].append(o.flatten(1).cpu())
        return hook

    for name, mod in model.hook_points().items():
        handles.append(mod.register_forward_hook(make_hook(name)))
    preds = [model(x[i:i + bs].to(dev)).argmax(1).cpu() for i in range(0, len(x), bs)]
    for h in handles:
        h.remove()
    pooled = {k: torch.cat(v) for k, v in pooled.items()}
    flat = {k: torch.cat(v) for k, v in flat.items()}
    return flat, pooled, torch.cat(preds)


def linear_cka(X, Y):
    X = X.to(dev).double(); Y = Y.to(dev).double()
    X = X - X.mean(0, keepdim=True)
    Y = Y - Y.mean(0, keepdim=True)
    K, L = X @ X.T, Y @ Y.T
    hsic = (K * L).sum()
    denom = torch.sqrt((K * K).sum() * (L * L).sum()) + 1e-12
    return (hsic / denom).item()


def layer_metrics(clean_flat, adv_flat, success_mask):
    out = {}
    for k in LAYERS:
        c, a = clean_flat[k], adv_flat[k]
        cos = F.cosine_similarity(c, a, dim=1)
        rel = (a - c).norm(dim=1) / (c.norm(dim=1) + 1e-12)
        m = {"cos_all": cos.mean().item(),
             "rel_l2_all": rel.mean().item(),
             "cka_all": linear_cka(c, a)}
        if success_mask.sum() > 0:
            m["cos_success"] = cos[success_mask].mean().item()
            m["rel_l2_success"] = rel[success_mask].mean().item()
        if (~success_mask).sum() > 0:
            m["cos_fail"] = cos[~success_mask].mean().item()
            m["rel_l2_fail"] = rel[~success_mask].mean().item()
        out[k] = m
    return out


def probe_auroc(clean_pooled, adv_pooled, seed=SEED):
    """Linear probe per layer: clean (0) vs adversarial (1). Split is by image."""
    n = len(next(iter(clean_pooled.values())))
    perm = np.random.RandomState(seed).permutation(n)
    split = int(0.6 * n)
    tr, te = perm[:split], perm[split:]
    feats = {k: (clean_pooled[k].numpy(), adv_pooled[k].numpy()) for k in LAYERS}
    feats["concat"] = (np.concatenate([clean_pooled[k].numpy() for k in LAYERS], axis=1),
                       np.concatenate([adv_pooled[k].numpy() for k in LAYERS], axis=1))
    res = {}
    for k, (c, a) in feats.items():
        Xtr = np.concatenate([c[tr], a[tr]]); ytr = np.r_[np.zeros(len(tr)), np.ones(len(tr))]
        Xte = np.concatenate([c[te], a[te]]); yte = np.r_[np.zeros(len(te)), np.ones(len(te))]
        sc = StandardScaler().fit(Xtr)
        clf = LogisticRegression(max_iter=2000, C=1.0).fit(sc.transform(Xtr), ytr)
        res[k] = float(roc_auc_score(yte, clf.predict_proba(sc.transform(Xte))[:, 1]))
    return res

In [11]:
clean_pred = predict(model, x_test)
clean_acc = (clean_pred == y_test).float().mean().item()
print(f"clean accuracy on eval subset: {clean_acc:.4f}")

correct_idx = (clean_pred == y_test).nonzero().squeeze(1)[:N_ANALYSIS]
assert len(correct_idx) >= 50, "Too few clean-correct images; train for more epochs."
xa, ya = x_test[correct_idx], y_test[correct_idx]
clean_flat, clean_pooled, _ = get_acts(model, xa)

results = {"clean_acc": clean_acc, "n_eval": len(x_test), "n_analysis": len(xa),
           "eps_255": EPS_255, "attacks": {}}

clean accuracy on eval subset: 0.3300


In [12]:
for name in ["fgsm", "pgd"]:
    results["attacks"][name] = {}
    for eps in EPS_LIST:
        t0 = time.time()
        # accuracy under attack on the whole eval subset
        adv_all = run_attack(model, x_test, y_test, name, eps)
        acc = (predict(model, adv_all) == y_test).float().mean().item()

        # layerwise analysis on clean-correct images
        adv = run_attack(model, xa, ya, name, eps)
        adv_flat, adv_pooled, adv_pred_a = get_acts(model, adv)
        success = adv_pred_a != ya

        results["attacks"][name][str(eps)] = {
            "acc": acc,
            "success_rate": success.float().mean().item(),
            "linf_check": (adv - xa).abs().max().item(),
            "layers": layer_metrics(clean_flat, adv_flat, success),
            "probe_auroc": probe_auroc(clean_pooled, adv_pooled),
        }
        print(f"{name} eps={eps * 255:g}/255: acc {acc:.3f}, success {success.float().mean():.3f}, "
              f"max|delta| {(adv - xa).abs().max().item() * 255:.2f}/255, {time.time() - t0:.0f}s")
        del adv_flat

with open(os.path.join(OUT, "results.json"), "w") as f:
    json.dump(results, f, indent=2)

fgsm eps=1/255: acc 0.095, success 0.712, max|delta| 1.00/255, 9s
fgsm eps=2/255: acc 0.030, success 0.909, max|delta| 2.00/255, 6s
fgsm eps=4/255: acc 0.015, success 0.955, max|delta| 4.00/255, 7s
fgsm eps=8/255: acc 0.010, success 0.970, max|delta| 8.00/255, 6s
pgd eps=1/255: acc 0.045, success 0.864, max|delta| 1.00/255, 48s
pgd eps=2/255: acc 0.000, success 1.000, max|delta| 2.00/255, 44s
pgd eps=4/255: acc 0.000, success 1.000, max|delta| 4.00/255, 45s
pgd eps=8/255: acc 0.000, success 1.000, max|delta| 8.00/255, 44s


In [13]:
attacks = ["fgsm", "pgd"]

# 1. accuracy vs attack size
plt.figure(figsize=(5, 4))
plt.axhline(results["clean_acc"], color="k", ls="--", label="clean")
for a in attacks:
    plt.plot(EPS_255, [results["attacks"][a][str(e)]["acc"] for e in EPS_LIST], marker="o", label=a.upper())
plt.xlabel("epsilon (/255)"); plt.ylabel("accuracy"); plt.legend(); plt.title("Accuracy under attack")
plt.tight_layout(); plt.savefig(os.path.join(OUT, "accuracy_vs_eps.png"), dpi=150); plt.show()

# 2. layerwise divergence
metrics = [("cos_all", "cosine similarity"), ("rel_l2_all", "relative L2"), ("cka_all", "linear CKA")]
fig, axes = plt.subplots(2, 3, figsize=(13, 7))
for r, a in enumerate(attacks):
    for c, (m, title) in enumerate(metrics):
        ax = axes[r, c]
        for e in EPS_LIST:
            ax.plot(LAYERS, [results["attacks"][a][str(e)]["layers"][k][m] for k in LAYERS],
                    marker="o", label=f"eps={e * 255:g}/255")
        ax.set_title(f"{a.upper()}: {title}"); ax.set_xlabel("layer")
        if r == 0 and c == 0:
            ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(OUT, "layerwise_divergence.png"), dpi=150); plt.show()

# 3. probe AUROC
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, a in zip(axes, attacks):
    for e in EPS_LIST:
        p = results["attacks"][a][str(e)]["probe_auroc"]
        ax.plot(LAYERS + ["concat"], [p[k] for k in LAYERS + ["concat"]], marker="o", label=f"eps={e * 255:g}/255")
    ax.axhline(0.5, color="gray", ls=":"); ax.set_ylim(0.4, 1.02)
    ax.set_title(f"{a.upper()}: clean vs adversarial probe AUROC"); ax.set_xlabel("layer"); ax.legend(fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(OUT, "probe_auroc.png"), dpi=150); plt.show()

In [14]:
lines = [f"Clean test accuracy: {results['clean_acc']:.4f} (n={results['n_eval']})", "",
         "| attack | eps | accuracy | attack success rate |", "|---|---|---|---|"]
for a in attacks:
    for e in EPS_LIST:
        r = results["attacks"][a][str(e)]
        lines.append(f"| {a.upper()} | {e * 255:g}/255 | {r['acc']:.3f} | {r['success_rate']:.3f} |")
lines += ["", f"Layerwise results on {results['n_analysis']} clean-correct images.", ""]
for a in attacks:
    for e in EPS_LIST:
        r = results["attacks"][a][str(e)]
        lines += [f"**{a.upper()} eps={e * 255:g}/255**", "",
                  "| layer | cosine | rel L2 | CKA | probe AUROC |", "|---|---|---|---|---|"]
        for k in LAYERS:
            m = r["layers"][k]
            lines.append(f"| {k} | {m['cos_all']:.3f} | {m['rel_l2_all']:.3f} | {m['cka_all']:.3f} | {r['probe_auroc'][k]:.3f} |")
        lines += [f"| concat | | | | {r['probe_auroc']['concat']:.3f} |", ""]
summary = "\n".join(lines)
with open(os.path.join(OUT, "results_summary.md"), "w") as f:
    f.write(summary)
print(summary)

Clean test accuracy: 0.3300 (n=200)

| attack | eps | accuracy | attack success rate |
|---|---|---|---|
| FGSM | 1/255 | 0.095 | 0.712 |
| FGSM | 2/255 | 0.030 | 0.909 |
| FGSM | 4/255 | 0.015 | 0.955 |
| FGSM | 8/255 | 0.010 | 0.970 |
| PGD | 1/255 | 0.045 | 0.864 |
| PGD | 2/255 | 0.000 | 1.000 |
| PGD | 4/255 | 0.000 | 1.000 |
| PGD | 8/255 | 0.000 | 1.000 |

Layerwise results on 66 clean-correct images.

**FGSM eps=1/255**

| layer | cosine | rel L2 | CKA | probe AUROC |
|---|---|---|---|---|
| stem | 1.000 | 0.017 | 1.000 | 0.582 |
| layer1 | 1.000 | 0.031 | 1.000 | 0.620 |
| layer2 | 0.997 | 0.071 | 1.000 | 0.682 |
| layer3 | 0.991 | 0.131 | 0.999 | 0.815 |
| layer4 | 0.982 | 0.189 | 0.998 | 0.929 |
| concat | | | | 0.930 |

**FGSM eps=2/255**

| layer | cosine | rel L2 | CKA | probe AUROC |
|---|---|---|---|---|
| stem | 0.999 | 0.032 | 1.000 | 0.675 |
| layer1 | 0.998 | 0.058 | 1.000 | 0.683 |
| layer2 | 0.992 | 0.129 | 1.000 | 0.783 |
| layer3 | 0.975 | 0.223 | 0.998 | 0.908 